# Step 1: Data Cleanup

In this notebook, we build our first rental data panel for Catalonia

## Output Structure

We build a unified `pandas` DataFrame with the following schema:

`Code | Municipality | Year | Quarter | Prices | Contracts | Population | Migration | Income`

where:
- `Quarter` is one of `Q1`, `Q2`, `Q3`, `Q4`, or `Y` (for a yearly-only var).
- `Prices` is rent price in **euros per square meter**.
- `Contracts` is the **number of rent contracts**
- `Population` is a nested dictionary of the form:
```py
{
    "Total": {"Total": X, "Males": X, "Females": X},
    "Native": {"Total": X, "Males": X, "Females": X},
    "Foreign": {"Total": X, "Males": X, "Females": X},
}
```
- `Migration` is a nested dictionary of the form:
```py
{
    "Total": {"Total": X, "Males": X, "Females": X},
    "Native": {"Total": X, "Males": X, "Females": X},
    "Foreign": {"Total": X, "Males": X, "Females": X},
}
```
- Finally, `Income` is a dictionary of the form:
```py
{
    "Tax Base": X,
    "Tax Liability": X,
}
```

### Municipality "Code" Matching

`codes.csv` contains the 6-digit unique codes for each municipality acording to Idescat. Meanwhile, the rental and demographic data contains 5-digit codes and/or the plaintext names. In this notebook we use the 5-digit codes as the canonical key to match municipalities across datasets.

## Preamble

In [77]:
from __future__ import annotations

from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 160)

## Paths

Here we locate our project root, then store the dir of all datasets.

In [78]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
DATA_DIR = ROOT / "data"

CODES_PATH = DATA_DIR / "codes.csv"
PRICES_PATH = DATA_DIR / "rent-prices" / "prc.xlsx"
CONTRACTS_PATH = DATA_DIR / "rent-contracts" / "con.xlsx"
POP_PATH = DATA_DIR / "population" / "pop.csv"
MIG_DIR = DATA_DIR / "migration"
INC_PATH = DATA_DIR / "income" / "inc.csv"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
for p in [CODES_PATH, PRICES_PATH, CONTRACTS_PATH, POP_PATH, MIG_DIR, INC_PATH]:
    print(f"{p.name:>15} -> {'FOUND' if p.exists() else 'MISSING'}")

ROOT: /home/venatio/documents/university/public-sector-2026
DATA_DIR: /home/venatio/documents/university/public-sector-2026/data
      codes.csv -> FOUND
       prc.xlsx -> FOUND
       con.xlsx -> FOUND
        pop.csv -> FOUND
      migration -> FOUND
        inc.csv -> FOUND


## Helpers

In [79]:
# normalizes names to exclude "problematic" characters
def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))

    value = value.replace("’", "'").replace("`", "'")
    value = value.replace("-", " ")
    value = value.replace("/", " ")
    value = value.replace("'", "")

    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

# transforms empty time-series values into pd.nan
def safe_float(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    value = str(value).strip()
    if value in {"", "-", "—", "nan", "None"}:
        return np.nan

    # handle spanish decimal/thousand(s) conventions if they appear
    if re.fullmatch(r"\d{1,3}(\.\d{3})+(,\d+)?", value):
        value = value.replace(".", "")
    value = value.replace(",", ".")

    try:
        return float(value)
    except ValueError:
        return np.nan


def safe_int(value):
    out = safe_float(value)
    return pd.NA if pd.isna(out) else int(round(out))


def zfill_code(value, width: int):
    if pd.isna(value):
        return pd.NA
    digits = re.sub(r"\D", "", str(value))
    if digits == "":
        return pd.NA
    return digits.zfill(width)


QUARTER_MAP = {
    "I": "Q1",
    "II": "Q2",
    "III": "Q3",
    "IV": "Q4",
    "MITJANA": "Y",
    "TOTAL": "Y",
}


def clean_header_piece(x) -> str:
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.startswith("Unnamed:"):
        return ""
    return x


def normalize_header_key(x) -> str:
    return normalize_text(x).replace(" ", "")


def extract_year_from_text(text):
    match = re.search(r"(19|20)\d{2}", str(text))
    return int(match.group(0)) if match else None

# used later for migration data
def pack_triplet(total=np.nan, males=np.nan, females=np.nan) -> dict:
    return {
        "Total": None if pd.isna(total) else int(total),
        "Males": None if pd.isna(males) else int(males),
        "Females": None if pd.isna(females) else int(females),
    }

## Municipality `Code-Name` Column Generation

This flattens the 6-digit codes into the 5-digit cannonical key, and for municipalities with no apparent code it creates a normalized name key.

In [80]:
codes = pd.read_csv(CODES_PATH, skiprows=3, dtype=object)

# data is only 4 columns long so this seems like a redundant step, still for peace-of-mind
codes = codes.iloc[:, :4].copy()
codes.columns = ["code_idescat_6", "Municipality", "comarca_code", "Comarca"]

codes["code_idescat_6"] = codes["code_idescat_6"].map(lambda x: zfill_code(x, 6))
codes["Code"] = codes["code_idescat_6"].str[:5]
codes["municipality_norm"] = codes["Municipality"].map(normalize_text)

codes = codes[["Code", "Municipality", "Comarca", "code_idescat_6", "municipality_norm"]].copy()
codes = codes.dropna(subset=["Code", "Municipality"]).drop_duplicates(subset=["Code"])

print("Municipalities in crosswalk:", len(codes))
print("Unique 5-digit codes:", codes["Code"].nunique())
print("Unique normalized names:", codes["municipality_norm"].nunique())

dup_norm = codes[codes.duplicated("municipality_norm", keep=False)].sort_values("municipality_norm")
if len(dup_norm):
    print("\nPotential duplicate normalized municipality names to inspect manually:")
    display(dup_norm)
else:
    print("\nNo duplicate normalized municipality names detected in the crosswalk.")

codes.head()

Municipalities in crosswalk: 947
Unique 5-digit codes: 947
Unique normalized names: 947

No duplicate normalized municipality names detected in the crosswalk.


,Code,Municipality,Comarca,code_idescat_6,municipality_norm
0,25001,Abella de la Conca,Pallars Jussà,250019,abella de la conca
1,08001,Abrera,Baix Llobregat,080018,abrera
2,25002,Àger,Noguera,250024,ager
3,25003,Agramunt,Urgell,250030,agramunt
4,08002,Aguilar de Segarra,Bages,080023,aguilar de segarra


## Rent Prices

As these are `.xlsx` files, the structure has: (i) one sheet per year, (ii) data starts at row 6-7, (iii) we're interested on the accumulated quarterly panel.

In [81]:
def read_multiheader_sheet(path: Path, sheet_name: str, skiprows: int) -> pd.DataFrame:
    df = pd.read_excel(
        path,
        sheet_name=sheet_name,
        skiprows=skiprows,
        header=[0, 1],
        dtype=object,
    )
    df = df.dropna(how="all").copy()

    new_cols = []
    last_top = ""
    for top, low in df.columns:
        top = clean_header_piece(top)
        low = clean_header_piece(low)
        if top:
            last_top = top
        new_cols.append((last_top, low))

    df.columns = pd.MultiIndex.from_tuples(new_cols)
    return df


def parse_rent_workbook(path: Path, kind: str) -> pd.DataFrame:
    if kind not in {"prices", "contracts"}:
        raise ValueError("kind must be either 'prices' or 'contracts'")

    value_name = "Prices" if kind == "prices" else "Contracts"
    xls = pd.ExcelFile(path)
    pieces = []

    for sheet_name in xls.sheet_names:
        year = extract_year_from_text(sheet_name)
        if year is None:
            print(f"Skipping sheet without year in name: {sheet_name}")
            continue

        # header rules
        if kind == "prices":
            # 2016-2025 => start at row 5, 2005-2015 => start at row 4
            skiprows = 4 if year >= 2016 else 3
        else:
            # contracts headers start on row 4
            skiprows = 3

        raw = read_multiheader_sheet(path, sheet_name=sheet_name, skiprows=skiprows)

        col_code = None
        col_muni = None
        selected_value_cols = {}

        for col in raw.columns:
            top, low = col
            top_key = normalize_header_key(top)
            low_key = normalize_header_key(low)

            if low_key == "codi" and col_code is None:
                col_code = col
            elif low_key in {"municipi", "municipio"} and col_muni is None:
                col_muni = col

            # Ignore accumulated block entirely
            # if "acumulat" in top_key:
            #     continue

            # keep only quarter columns
            quarter_candidates = {
                "i": "Q1",
                "ii": "Q2",
                "iii": "Q3",
                "iv": "Q4",
            }

            if low_key in quarter_candidates:
                selected_value_cols[quarter_candidates[low_key]] = col

        if col_code is None or col_muni is None:
            raise ValueError(
                f"Could not identify municipality code/name columns in sheet '{sheet_name}'."
            )

        expected_quarters = {"Q1", "Q2", "Q3", "Q4"}
        missing = expected_quarters - set(selected_value_cols.keys())
        if missing:
            print(f"Warning: in {sheet_name}, missing quarter columns: {sorted(missing)}")

        keep_order = ["Q1", "Q2", "Q3", "Q4"]
        keep_cols = [col_code, col_muni] + [
            selected_value_cols[q] for q in keep_order if q in selected_value_cols
        ]

        df = raw[keep_cols].copy()
        df.columns = ["Code", "Municipality_source"] + [
            q for q in keep_order if q in selected_value_cols
        ]

        df["Code"] = df["Code"].map(lambda x: zfill_code(x, 5))
        df["Year"] = year

        value_cols = [c for c in ["Q1", "Q2", "Q3", "Q4"] if c in df.columns]
        df = df.melt(
            id_vars=["Code", "Municipality_source", "Year"],
            value_vars=value_cols,
            var_name="Quarter",
            value_name=value_name,
        )

        if kind == "prices":
            df["Prices"] = df["Prices"].map(safe_float)
        else:
            df["Contracts"] = df["Contracts"].map(safe_int)

        pieces.append(df)

    out = pd.concat(pieces, ignore_index=True)
    out = out.merge(codes[["Code", "Municipality"]], on="Code", how="left")
    out["Municipality"] = out["Municipality"].fillna(out["Municipality_source"])
    out = out.drop(columns=["Municipality_source"])
    out = out[["Code", "Municipality", "Year", "Quarter", value_name]]
    out = out.sort_values(["Code", "Year", "Quarter"]).reset_index(drop=True)
    return out

In [82]:
prices = parse_rent_workbook(PRICES_PATH, kind="prices")
contracts = parse_rent_workbook(CONTRACTS_PATH, kind="contracts")

print("Prices shape:", prices.shape)
print("Contracts shape:", contracts.shape)

Prices shape: (79684, 5)
Contracts shape: (79640, 5)


In [83]:
display(prices.head(10))
display(contracts.head(10))

,Code,Municipality,Year,Quarter,Prices
0,00005,NaN,2015,Q1,NaN
1,00005,NaN,2015,Q2,NaN
2,00005,NaN,2015,Q3,NaN
3,00005,NaN,2015,Q4,NaN
4,02050,NaN,2005,Q1,NaN
5,02050,NaN,2005,Q2,NaN
6,02050,NaN,2005,Q3,NaN
7,02050,NaN,2005,Q4,NaN
8,02050,NaN,2006,Q1,NaN
9,02050,NaN,2006,Q2,NaN


,Code,Municipality,Year,Quarter,Contracts
0,08001,Abrera,2005,Q1,8
1,08001,Abrera,2005,Q2,14
2,08001,Abrera,2005,Q3,19
3,08001,Abrera,2005,Q4,25
4,08001,Abrera,2006,Q1,4
5,08001,Abrera,2006,Q2,9
6,08001,Abrera,2006,Q3,14
7,08001,Abrera,2006,Q4,18
8,08001,Abrera,2007,Q1,11
9,08001,Abrera,2007,Q2,18


## Population

Here we define `Native` as `Catalonia + rest of the State`, and we keep only the `{Native, Foreign, Total}` count, broken down into gender as `{Total, Male, Female}`.

In [84]:
pop = pd.read_csv(POP_PATH, dtype=object)
pop.columns = [c.strip() for c in pop.columns]

pop["year"] = pd.to_numeric(pop["year"], errors="coerce").astype("Int64")
pop["value"] = pd.to_numeric(pop["value"], errors="coerce")

pop["municipality_norm"] = pop["municipalities"].map(normalize_text)
pop["sex_key"] = pop["sex"].map(lambda x: {"males": "Males", "females": "Females", "total": "Total"}.get(normalize_text(x)))
pop["origin_key"] = pop["place of birth"].map(
    lambda x: {
        "catalonia": "Native",
        "rest of the state": "Native",
        "foreign": "Foreign",
        "total": "Total",
    }.get(normalize_text(x))
)

pop = pop[
    (pop["concept"].map(normalize_text) == "population")
    & pop["sex_key"].notna()
    & pop["origin_key"].notna()
].copy()

pop_grouped = pop.groupby(["year", "municipality_norm", "origin_key", "sex_key"], as_index=False)["value"].sum(min_count=1)

pop_pivot = (
    pop_grouped.pivot_table(
        index=["year", "municipality_norm"],
        columns=["origin_key", "sex_key"],
        values="value",
        aggfunc="sum",
    )
    .sort_index(axis=1)
)

expected_pop_cols = pd.MultiIndex.from_product(
    [["Total", "Native", "Foreign"], ["Total", "Males", "Females"]]
)
for col in expected_pop_cols:
    if col not in pop_pivot.columns:
        pop_pivot[col] = np.nan

pop_pivot = pop_pivot[expected_pop_cols].reset_index()

def build_population_dict(row) -> dict:
    return {
        "Total": pack_triplet(row[("Total", "Total")], row[("Total", "Males")], row[("Total", "Females")]),
        "Native": pack_triplet(row[("Native", "Total")], row[("Native", "Males")], row[("Native", "Females")]),
        "Foreign": pack_triplet(row[("Foreign", "Total")], row[("Foreign", "Males")], row[("Foreign", "Females")]),
    }

population = pop_pivot.copy()
population[("meta", "Population")] = population.apply(build_population_dict, axis=1)

population = pd.DataFrame({
    "year": population[("year", "")],
    "municipality_norm": population[("municipality_norm", "")],
    "Population": population[("meta", "Population")],
})

# FIX: these have mismatched names
manual_population_name_fixes = {
    "bisbal de falset la": "bisbal de montsant la",
    "castell platja daro": "castell daro platja daro i sagaro",
    "masarac": "masarac i vilarnadal",
}

# FIX: Catalunya is aggregate, and Medinya got swallowed
drop_population_norms = {
    "catalunya",
    "medinya",
}

# apply the manual fixes identified above
population["municipality_norm"] = population["municipality_norm"].replace(manual_population_name_fixes)
population = population[~population["municipality_norm"].isin(drop_population_norms)].copy()

population = population.merge(
    codes[["Code", "Municipality", "municipality_norm"]],
    on="municipality_norm",
    how="left",
)

unmatched_population = population[population["Code"].isna()][["year", "municipality_norm"]].drop_duplicates().sort_values(["year", "municipality_norm"])
print("Unmatched population municipality-years:", len(unmatched_population))
if len(unmatched_population):
    display(unmatched_population.head(25))

population = population[population["Code"].notna()].copy()
population["Year"] = population["year"].astype(int)
population["Quarter"] = "Y"

population = population[["Code", "Municipality", "Year", "Quarter", "Population"]].sort_values(["Code", "Year"])

Unmatched population municipality-years: 0


In [85]:
population.head()

,Code,Municipality,Year,Quarter,Population
1,08001,Abrera,2000,Y,"{'Total': {'Total': 8177, 'Males': 4203, 'Fema..."
948,08001,Abrera,2001,Y,"{'Total': {'Total': 8454, 'Males': 4363, 'Fema..."
1895,08001,Abrera,2002,Y,"{'Total': {'Total': 8823, 'Males': 4562, 'Fema..."
2842,08001,Abrera,2003,Y,"{'Total': {'Total': 9166, 'Males': 4753, 'Fema..."
3789,08001,Abrera,2004,Y,"{'Total': {'Total': 9422, 'Males': 4890, 'Fema..."


## Migration

We have `Total` from `Saldo Total`, `Inflows` from `Saldo Interior`, and `Outflows` from `Saldo Exterior`.

In [86]:
def find_column_by_alias(df: pd.DataFrame, aliases: set[str]) -> str | None:
    norm_map = {col: normalize_text(col) for col in df.columns}
    for raw_col, norm_col in norm_map.items():
        if norm_col in aliases:
            return raw_col
    return None


def read_csv_flexible(path: Path) -> pd.DataFrame:
    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, sep=";", dtype=object, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path.name}")


def infer_year_column(df: pd.DataFrame):
    preferred = ["year", "año", "anio", "any", "period", "periodo", "fecha"]
    norm_map = {col: normalize_text(col) for col in df.columns}

    for raw_col, norm_col in norm_map.items():
        if any(token == norm_col or token in norm_col for token in preferred):
            extracted = df[raw_col].astype(str).str.extract(r"((?:19|20)\d{2})")[0]
            if extracted.notna().any():
                return raw_col
    return None


def parse_single_migration_file(path: Path, valid_codes: set) -> pd.DataFrame:
    df = read_csv_flexible(path)
    df.columns = [c.strip() for c in df.columns]

    municipio_col = find_column_by_alias(df, {"municipio", "municipios"})
    tipo_col = find_column_by_alias(df, {"tipo de variacion", "tipo de saldo"})
    nac_col = find_column_by_alias(df, {"nacionalidad"})
    sex_col = find_column_by_alias(df, {"sexo"})
    val_col = find_column_by_alias(df, {"total"})

    if municipio_col is None:
        raise ValueError(f"{path.name}: could not identify municipality column.")
    if tipo_col is None:
        raise ValueError(f"{path.name}: could not identify migration type column.")
    if nac_col is None:
        raise ValueError(f"{path.name}: could not identify nationality column.")
    if sex_col is None:
        raise ValueError(f"{path.name}: could not identify sex column.")
    if val_col is None:
        raise ValueError(f"{path.name}: could not identify value column.")

    df["Code"] = df[municipio_col].astype(str).str.extract(r"^(\d{5})")[0]

    years_in_name = re.findall(r"(?:19|20)\d{2}", path.stem)
    if len(set(years_in_name)) == 1:
        df["Year"] = int(years_in_name[0])
    else:
        year_col = infer_year_column(df)
        if year_col is None:
            raise ValueError(
                f"{path.name}: filename spans multiple years but no year column was found. "
                "Inspect this file and extend the parser for its exact layout."
            )
        df["Year"] = pd.to_numeric(
            df[year_col].astype(str).str.extract(r"((?:19|20)\d{2})")[0],
            errors="coerce",
        ).astype("Int64")

    df = df[df["Code"].isin(valid_codes)].copy()

    tipo_map = {
        "saldo total": "Total",
        "saldo interior": "Inflows",
        "saldo exterior": "Outflows",
    }
    sex_map = {
        "total": "Total",
        "hombres": "Males",
        "mujeres": "Females",
    }

    df["migration_key"] = df[tipo_col].map(lambda x: tipo_map.get(normalize_text(x)))
    df["sex_key"] = df[sex_col].map(lambda x: sex_map.get(normalize_text(x)))
    df["nac_key"] = df[nac_col].map(normalize_text)
    df[val_col] = pd.to_numeric(df[val_col], errors="coerce")

    df = df[
        df["migration_key"].notna()
        & df["sex_key"].notna()
        & (df["nac_key"] == "total")
        & df["Year"].notna()
    ].copy()

    df["Year"] = df["Year"].astype(int)

    out = (
        df.groupby(["Code", "Year", "migration_key", "sex_key"], as_index=False)[val_col]
        .sum(min_count=1)
        .rename(columns={val_col: "value"})
    )
    return out


valid_codes = set(codes["Code"])
migration_files = sorted(MIG_DIR.glob("mig-*.csv"))

migration_long_parts = []
for path in migration_files:
    try:
        migration_long_parts.append(parse_single_migration_file(path, valid_codes))
        print(f"Parsed: {path.name}")
    except Exception as exc:
        print(f"Skipped/failed: {path.name} -> {exc}")

migration_long = pd.concat(migration_long_parts, ignore_index=True)

migration_long = (
    migration_long.groupby(["Code", "Year", "migration_key", "sex_key"], as_index=False)["value"]
    .sum(min_count=1)
)

migration_pivot = migration_long.pivot_table(
    index=["Code", "Year"],
    columns=["migration_key", "sex_key"],
    values="value",
    aggfunc="sum",
)

expected_mig_cols = pd.MultiIndex.from_product(
    [["Total", "Inflows", "Outflows"], ["Total", "Males", "Females"]]
)
for col in expected_mig_cols:
    if col not in migration_pivot.columns:
        migration_pivot[col] = np.nan

migration_pivot = migration_pivot[expected_mig_cols].reset_index()

def build_migration_dict(row) -> dict:
    return {
        "Total": pack_triplet(
            row[("Total", "Total")],
            row[("Total", "Males")],
            row[("Total", "Females")],
        ),
        "Inflows": pack_triplet(
            row[("Inflows", "Total")],
            row[("Inflows", "Males")],
            row[("Inflows", "Females")],
        ),
        "Outflows": pack_triplet(
            row[("Outflows", "Total")],
            row[("Outflows", "Males")],
            row[("Outflows", "Females")],
        ),
    }

migration = migration_pivot.copy()
migration[("meta", "Migration")] = migration.apply(build_migration_dict, axis=1)

migration = pd.DataFrame({
    "Code": migration[("Code", "")],
    "Year": migration[("Year", "")],
    "Migration": migration[("meta", "Migration")],
})

migration = migration.merge(
    codes[["Code", "Municipality"]],
    on="Code",
    how="left",
)

migration["Quarter"] = "Y"
migration = migration[["Code", "Municipality", "Year", "Quarter", "Migration"]].sort_values(["Code", "Year"])

Parsed: mig-2011.csv
Parsed: mig-2012.csv
Parsed: mig-2013.csv
Parsed: mig-2014.csv
Parsed: mig-2015.csv
Parsed: mig-2016.csv
Parsed: mig-2017.csv
Parsed: mig-2018.csv
Parsed: mig-2019.csv
Parsed: mig-2020.csv
Parsed: mig-2021.csv


In [87]:
migration.head()

,Code,Municipality,Year,Quarter,Migration
0,08001,Abrera,2011,Y,"{'Total': {'Total': 173, 'Males': 95, 'Females..."
1,08001,Abrera,2012,Y,"{'Total': {'Total': 103, 'Males': None, 'Femal..."
2,08001,Abrera,2013,Y,"{'Total': {'Total': -27, 'Males': None, 'Femal..."
3,08001,Abrera,2014,Y,"{'Total': {'Total': -98, 'Males': None, 'Femal..."
4,08001,Abrera,2015,Y,"{'Total': {'Total': 90, 'Males': None, 'Female..."


## Income

In [88]:
income = pd.read_csv(INC_PATH, dtype=object)
income.columns = [c.strip() for c in income.columns]

income["year"] = pd.to_numeric(income["year"], errors="coerce").astype("Int64")
income["value"] = pd.to_numeric(income["value"], errors="coerce")
income["municipality_norm"] = income["municipality"].map(normalize_text)
income["concept_key"] = income["concept"].map(
    lambda x: {
        "general taxable income per declarant": "Tax Base",
        "tax liability as declared on self assessment return per declarant": "Tax Liability",
    }.get(normalize_text(x).replace("€", "").strip())
)

income = income[
    income["year"].notna()
    & income["municipality_norm"].notna()
    & income["concept_key"].notna()
].copy()

# FIX: same name fixes / drops pattern as population
manual_income_name_fixes = {
    "bisbal de falset la": "bisbal de montsant la",
    "castell platja daro": "castell daro platja daro i sagaro",
    "masarac": "masarac i vilarnadal",
}
drop_income_norms = {"catalunya", "medinya"}

income["municipality_norm"] = income["municipality_norm"].replace(manual_income_name_fixes)
income = income[~income["municipality_norm"].isin(drop_income_norms)].copy()

income_grouped = (
    income.groupby(["year", "municipality_norm", "concept_key"], as_index=False)["value"]
    .sum(min_count=1)
)

income_pivot = (
    income_grouped.pivot_table(
        index=["year", "municipality_norm"],
        columns="concept_key",
        values="value",
        aggfunc="sum",
    )
    .reset_index()
)

for col in ["Tax Base", "Tax Liability"]:
    if col not in income_pivot.columns:
        income_pivot[col] = np.nan

def build_income_dict(row) -> dict:
    return {
        "Tax Base": None if pd.isna(row["Tax Base"]) else float(row["Tax Base"]),
        "Tax Liability": None if pd.isna(row["Tax Liability"]) else float(row["Tax Liability"]),
    }

income_annual = income_pivot.copy()
income_annual["Income"] = income_annual.apply(build_income_dict, axis=1)

income_annual = income_annual.merge(
    codes[["Code", "Municipality", "municipality_norm"]],
    on="municipality_norm",
    how="left",
)

unmatched_income = (
    income_annual[income_annual["Code"].isna()][["year", "municipality_norm"]]
    .drop_duplicates()
    .sort_values(["year", "municipality_norm"])
)
print("Unmatched income municipality-years:", len(unmatched_income))
if len(unmatched_income):
    display(unmatched_income.head(25))

income_annual = income_annual[income_annual["Code"].notna()].copy()
income_annual["Year"] = income_annual["year"].astype(int)
income_annual["Quarter"] = "Y"

income_annual = (
    income_annual[["Code", "Municipality", "Year", "Quarter", "Income"]]
    .sort_values(["Code", "Year"])
    .reset_index(drop=True)
)

Unmatched income municipality-years: 0


In [89]:
income_annual.head()

,Code,Municipality,Year,Quarter,Income
0,08001,Abrera,2000,Y,"{'Tax Base': 11040.0, 'Tax Liability': 2255.0}"
1,08001,Abrera,2001,Y,"{'Tax Base': 11652.0, 'Tax Liability': 2371.0}"
2,08001,Abrera,2002,Y,"{'Tax Base': 12372.0, 'Tax Liability': 2581.0}"
3,08001,Abrera,2003,Y,"{'Tax Base': 15696.0, 'Tax Liability': 2632.0}"
4,08001,Abrera,2004,Y,"{'Tax Base': 16123.0, 'Tax Liability': 2744.0}"


## Build Panel

Here we build the unified panel using a `code-muni-year-quarter` key.

In [90]:
base_keys = pd.concat(
    [
        prices[["Code", "Municipality", "Year", "Quarter"]],
        contracts[["Code", "Municipality", "Year", "Quarter"]],
        population[["Code", "Municipality", "Year", "Quarter"]],
        migration[["Code", "Municipality", "Year", "Quarter"]],
        income_annual[["Code", "Municipality", "Year", "Quarter"]],
    ],
    ignore_index=True,
).drop_duplicates()

master_mixed = (
    base_keys
    .merge(prices, on=["Code", "Municipality", "Year", "Quarter"], how="left")
    .merge(contracts, on=["Code", "Municipality", "Year", "Quarter"], how="left")
    .merge(population, on=["Code", "Municipality", "Year", "Quarter"], how="left")
    .merge(migration, on=["Code", "Municipality", "Year", "Quarter"], how="left")
    .merge(income_annual, on=["Code", "Municipality", "Year", "Quarter"], how="left")
)

quarter_order = pd.CategoricalDtype(categories=["Q1", "Q2", "Q3", "Q4", "Y"], ordered=True)
master_mixed["Quarter"] = master_mixed["Quarter"].astype(quarter_order)

master_mixed = master_mixed.sort_values(["Code", "Year", "Quarter"]).reset_index(drop=True)

In [91]:
master_mixed.head(20)

,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income
0,00005,NaN,2015,Q1,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,2015,Q2,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,2015,Q3,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,2015,Q4,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2005,Q1,NaN,NaN,NaN,NaN,NaN
5,02050,NaN,2005,Q2,NaN,NaN,NaN,NaN,NaN
6,02050,NaN,2005,Q3,NaN,NaN,NaN,NaN,NaN
7,02050,NaN,2005,Q4,NaN,NaN,NaN,NaN,NaN
8,02050,NaN,2006,Q1,NaN,NaN,NaN,NaN,NaN
9,02050,NaN,2006,Q2,NaN,NaN,NaN,NaN,NaN


## Data Validationn

Here we check that (i) there's only one row per `code-year-quarter` key, (ii) that no municipalities are missing, (iii) annual rows are "adequately" covered.

In [92]:
master_2011 = master_mixed[master_mixed["Year"] >= 2011]

dup_count = master_2011.duplicated(["Code", "Year", "Quarter"]).sum()
print("Duplicate Code-Year-Quarter rows:", dup_count)

missing_muni = master_2011["Municipality"].isna().sum()
print("Rows with no municipality name:", missing_muni)

coverage = (
    master_2011.groupby("Quarter")[["Prices", "Contracts"]]
    .agg(lambda s: s.notna().sum())
    .rename(columns={
        "Prices": "Prices_non_null",
        "Contracts": "Contracts_non_null"
    })
)
display(coverage)

annual_covariates = (
    master_2011[master_2011["Quarter"] == "Y"][["Population", "Migration", "Income"]]
    .notna()
    .sum()
)

print("\nAnnual non-null counts:")
display(annual_covariates.to_frame("non_null_rows"))

Duplicate Code-Year-Quarter rows: 0
Rows with no municipality name: 80


,Prices_non_null,Contracts_non_null
Quarter,,
Q1,14207,14207
Q2,14207,14207
Q3,14207,14207
Q4,13260,13260
Y,0,0



Annual non-null counts:


,non_null_rows
Population,11364
Migration,10417
Income,12311


In [93]:
# shows a random subset of the data, uses `42` to make it persistent
master_2011.sample(min(len(master_2011), 10), random_state=42)

,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income
45612,17119,Palau de Santa Eulàlia,2022,Q3,0.000000,0,NaN,NaN,NaN
84783,43022,Batea,2015,Y,NaN,NaN,"{'Total': {'Total': 1945, 'Males': 966, 'Femal...","{'Total': {'Total': 36, 'Males': None, 'Female...","{'Tax Base': 11063.0, 'Tax Liability': 2603.0}"
27625,08258,Santa Maria d'Oló,2012,Q2,0.000000,4,NaN,NaN,NaN
7682,08071,Copons,2019,Q4,0.000000,3,NaN,NaN,NaN
63941,25064,Castellar de la Ribera,2016,Q2,0.000000,0,NaN,NaN,NaN
22913,08214,Vilassar de Dalt,2020,Q2,798.208333,48,NaN,NaN,NaN
67944,25112,Ivars de Noguera,2017,Q4,0.000000,0,NaN,NaN,NaN
12504,08116,"Masies de Roda, les",2012,Q1,0.000000,1,NaN,NaN,NaN
50425,17168,Sant Joan de Mollet,2013,Q1,0.000000,1,NaN,NaN,NaN
58007,25005,Alàs i Cerc,2017,Q3,0.000000,0,NaN,NaN,NaN


## Quarterly Panel

Here we take `Quarter == Y` rows and bring them forward into all quarters so as to create a balanced panel.

In [94]:
master_mixed = master_mixed[master_mixed["Year"] >= 2011].copy()

annual_covars = (
    master_mixed.loc[
        master_mixed["Quarter"] == "Y",
        ["Code", "Municipality", "Year", "Population", "Migration", "Income"],
    ]
    .drop_duplicates(subset=["Code", "Municipality", "Year"], keep="first")
    .copy()
)

quarterly_panel = (
    master_mixed.loc[master_mixed["Quarter"].isin(["Q1", "Q2", "Q3", "Q4"])].copy()
    .drop(columns=["Population", "Migration", "Income"])
    .merge(annual_covars, on=["Code", "Municipality", "Year"], how="left")
    .sort_values(["Code", "Year", "Quarter"])
    .reset_index(drop=True)
)

In [95]:
quarterly_panel.head(12)

,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income
0,00005,NaN,2015,Q1,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,2015,Q2,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,2015,Q3,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,2015,Q4,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2011,Q1,NaN,NaN,NaN,NaN,NaN
5,02050,NaN,2011,Q2,NaN,NaN,NaN,NaN,NaN
6,02050,NaN,2011,Q3,NaN,NaN,NaN,NaN,NaN
7,02050,NaN,2011,Q4,NaN,NaN,NaN,NaN,NaN
8,02050,NaN,2012,Q1,NaN,NaN,NaN,NaN,NaN
9,02050,NaN,2012,Q2,NaN,NaN,NaN,NaN,NaN


## Save Outputs

We store it in `pickle` formats to reuse them later.

In [23]:
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

master_mixed.to_pickle(OUT_DIR / "master_mixed.pkl")
quarterly_panel.to_pickle(OUT_DIR / "quarterly_panel.pkl")

print("Saved outputs to:", OUT_DIR)

Saved outputs to: /home/venatio/documents/university/public-sector-2026/out
